In [1]:
# ===================
# Imports and Setup
# ===================

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [2]:
# -----------------------------
# Load Notebook One Artifacts
# -----------------------------

df = pd.read_csv("rag_base_dataset.csv")
embeddings = np.load("essay_embeddings.npy")

In [4]:
# ==========================
# Recreate Feature Pipeline
# ==========================

def vocab_richness(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

df["word_count"] = df["full_text"].astype(str).apply(lambda x: len(x.split()))
df["vocab_richness"] = df["full_text"].astype(str).apply(vocab_richness)

feature_cols = ["word_count", "vocab_richness"]
X_features = df[feature_cols].fillna(0)

scaler = MinMaxScaler()
X_features_scaled = scaler.fit_transform(X_features)

# -------------------
# Retrieval Model
# -------------------

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
# ========================
# Retrieval Function
# ========================

def retrieve_similar_essays(new_text, top_k=5):

    new_embedding = model.encode([new_text])
    semantic_sim = cosine_similarity(new_embedding, embeddings)[0]

    new_word_count = len(new_text.split())
    new_vocab = vocab_richness(new_text)

    new_features_df = pd.DataFrame(
      [[new_word_count, new_vocab]],
      columns=feature_cols
    )

    new_features = scaler.transform(new_features_df)
    feature_sim = cosine_similarity(new_features, X_features_scaled)[0]

    # UPDATED WEIGHTS
    final_score = 0.6 * semantic_sim + 0.4 * feature_sim

    top_indices = np.argsort(final_score)[-top_k:][::-1]

    results = df.iloc[top_indices].copy()
    results["semantic_similarity"] = semantic_sim[top_indices]
    results["feature_similarity"] = feature_sim[top_indices]
    results["final_score"] = final_score[top_indices]

    return results

In [16]:
# ======================
# Core RAG Engine
# ======================

# This is where the fun begins!

# ------------------
# Helper Functions
# ------------------

# Extract Writing Signals
def analyze_essay(text):
    sentences = text.split(".")

    return {
        "length": len(text.split()),
        "sentences": len(sentences),
        "avg_sentence_length": len(text.split()) / max(len(sentences), 1),
        "vocab_richness": vocab_richness(text)
    }


# Compare to high-scoring Essays
def compare_to_top_essays(query_text, retrieved_df):

    query_stats = analyze_essay(query_text)

    high_score_df = retrieved_df[retrieved_df["score"] >= 4]

    if len(high_score_df) == 0:
        return "Not enough high-scoring essays to compare."

    avg_word_count = high_score_df["word_count"].mean()
    avg_vocab = high_score_df["vocab_richness"].mean()

    feedback = []

    if query_stats["length"] < avg_word_count:
        feedback.append("Your essay is shorter than higher-scoring essays. Consider expanding your ideas.")

    if query_stats["vocab_richness"] < avg_vocab:
        feedback.append("Try using more varied vocabulary to strengthen your writing.")

    if query_stats["avg_sentence_length"] < 10:
        feedback.append("Your sentences are quite short. Try combining ideas for more complex structure.")

    return feedback


In [20]:
# ==========================================
# RAG Interpretation Modalities
# ==========================================
# Responses are pulled from library
# Stochastic phrase selection
# Simulates LLM response from context
# There are four modes to this RAG model
# Mode One: Feedback Generator
# Mode Two: Grade Explanation
# Mode Three: Improvement Suggestions
# Mode Four: Simple Auto Scoring
# ===========================================

# ---------------------------
# Phrase Pool Implementation
# ---------------------------

# Create Phrase Pool
import random

FEEDBACK_LIBRARY = {
    "length_short": [
        "Your essay is shorter than stronger examples—try expanding your ideas.",
        "Consider adding more detail to match higher-scoring essays.",
        "Your response could benefit from more development and depth."
    ],

    "low_vocab": [
        "Try using more varied vocabulary to strengthen your writing.",
        "Your word choice is somewhat repetitive—consider diversifying it.",
        "Using a wider range of vocabulary could improve your essay."
    ],

    "good": [
        "You're on the right track and close to stronger responses.",
        "This essay shows solid understanding but could be refined further.",
        "You have a good foundation—just a few improvements needed."
    ],

    "weak": [
        "This response aligns more with lower-scoring essays.",
        "Your essay may need significant improvement to reach higher levels.",
        "There are several areas that need development."
    ]
}


# Mode One
def generate_feedback(query_text):

    retrieved = retrieve_similar_essays(query_text)
    query_stats = analyze_essay(query_text)

    feedback = []

    avg_score = retrieved["score"].mean()

    # Score-based feedback
    if avg_score < 3:
        feedback.append(random.choice(FEEDBACK_LIBRARY["weak"]))
    elif avg_score < 4:
        feedback.append(random.choice(FEEDBACK_LIBRARY["good"]))
    else:
        feedback.append("This essay is similar to high-scoring responses.")

    # Compare to high-scoring essays
    high_score_df = retrieved[retrieved["score"] >= 4]

    if len(high_score_df) > 0:
        avg_word_count = high_score_df["word_count"].mean()
        avg_vocab = high_score_df["vocab_richness"].mean()

        diff = avg_word_count - query_stats["length"]

        if diff > 50:
            feedback.append(f"Your essay is about {int(diff)} words shorter than strong examples.")
            feedback.append(random.choice(FEEDBACK_LIBRARY["length_short"]))
        elif diff > 20:
            feedback.append("Your essay is slightly shorter than higher-scoring responses.")

        if query_stats["vocab_richness"] < avg_vocab:
            feedback.append(
                f"Your vocabulary richness ({query_stats['vocab_richness']:.2f}) "
                f"is below stronger essays (~{avg_vocab:.2f})."
            )
            feedback.append(random.choice(FEEDBACK_LIBRARY["low_vocab"]))

    return {
        "retrieved": retrieved,
        "feedback": feedback
    }

# Mode Two
def explain_grade(query_text):

    retrieved = retrieve_similar_essays(query_text)

    explanation = []

    explanation.append(
        f"Your essay is most similar to essays scoring between "
        f"{retrieved['score'].min()} and {retrieved['score'].max()}."
    )

    explanation.append(
        f"The average similarity-based score is approximately "
        f"{retrieved['score'].mean():.2f}."
    )

    explanation.append(
        "This suggests your writing level aligns with these examples "
        "in both content and structure."
    )

    return explanation


# Mode Three
def suggest_improvements(query_text):

    retrieved = retrieve_similar_essays(query_text)
    query_stats = analyze_essay(query_text)

    suggestions = []

    high_score_df = retrieved[retrieved["score"] >= 5]

    if len(high_score_df) == 0:
        return ["No top-tier examples found. Try expanding your essay."]

    avg_word_count = high_score_df["word_count"].mean()
    avg_vocab = high_score_df["vocab_richness"].mean()

    # Targeted improvements
    if query_stats["length"] < avg_word_count:
        suggestions.append("Expand your essay to include more detailed arguments and examples.")

    if query_stats["vocab_richness"] < avg_vocab:
        suggestions.append("Incorporate more varied and precise vocabulary.")

    if query_stats["avg_sentence_length"] < 10:
        suggestions.append("Try combining sentences to create more complex and fluid writing.")

    # General high-score traits
    suggestions.append("Top-scoring essays tend to:")
    suggestions.append("- Present clear and well-supported arguments")
    suggestions.append("- Use specific and relevant examples")
    suggestions.append("- Maintain strong organization across paragraphs")

    return suggestions


# Mode Four
def predict_score(query_text):

    retrieved = retrieve_similar_essays(query_text)

    return float(round(retrieved["score"].mean(), 2))

# ----------------------------------------
# Paragraph Builder (Final Output Layer)
# ----------------------------------------

def build_paragraph(feedback_list):

    intro_options = [
        "Here's some feedback on your essay:",
        "After reviewing similar essays, here's what stands out:",
        "Based on comparable responses, here are some suggestions:"
    ]

    intro = random.choice(intro_options)

    paragraph = intro + "\n\n"

    for f in feedback_list:
        paragraph += f"- {f}\n"

    return paragraph

In [21]:
# ==================
# Unified Interface
# ==================

def rag_system(query_text, mode="feedback"):

    if mode == "feedback":
        result = generate_feedback(query_text)
        return result

    elif mode == "explain":
        return explain_grade(query_text)

    elif mode == "improve":
        return suggest_improvements(query_text)

    elif mode == "score":
        return predict_score(query_text)

    else:
        return "Invalid mode."

In [22]:
# ------------
# System Test
# ------------

# Sampel Essay
test_essay = df["full_text"].iloc[10]

# Feedback Mode
result = rag_system(test_essay, mode="feedback")

for f in result["feedback"]:
    print("-", f)

# Grade Explanation Mode
rag_system(test_essay, mode="explain")

# Improvement Suggestions Mode
rag_system(test_essay, mode="improve")

# Score Predictions Mode
rag_system(test_essay, mode="score")

- This essay shows solid understanding but could be refined further.
- Your vocabulary richness (0.52) is below stronger essays (~0.55).
- Using a wider range of vocabulary could improve your essay.


3.4

In [23]:
# ==========================================
# RAG SYSTEM SANITY CHECK
# ==========================================

def run_sanity_check(sample_text):

    print("="*60)
    print("RUNNING RAG SYSTEM SANITY CHECK")
    print("="*60)

    # -----------------------
    # FEEDBACK MODE
    # -----------------------
    print("\n[1] Testing Feedback Mode...")

    feedback_result = rag_system(sample_text, mode="feedback")

    assert isinstance(feedback_result, dict), "Feedback output should be a dictionary"
    assert "feedback" in feedback_result, "Missing 'feedback' key"
    assert isinstance(feedback_result["feedback"], list), "Feedback should be a list"
    assert len(feedback_result["feedback"]) > 0, "Feedback list is empty"

    print("✔ Feedback structure valid")

    # Build paragraph
    paragraph = build_paragraph(feedback_result["feedback"])

    assert isinstance(paragraph, str), "Paragraph output should be a string"
    assert len(paragraph) > 20, "Paragraph too short"

    print("✔ Paragraph generated successfully\n")
    print("----- Generated Paragraph -----")
    print(paragraph)

    # -----------------------
    # EXPLAIN MODE
    # -----------------------
    print("\n[2] Testing Grade Explanation...")

    explanation = rag_system(sample_text, mode="explain")

    assert isinstance(explanation, list), "Explanation should be a list"
    assert len(explanation) > 0, "Explanation is empty"

    print("✔ Explanation working")
    print(explanation)

    # -----------------------
    # IMPROVEMENT MODE
    # -----------------------
    print("\n[3] Testing Improvement Suggestions...")

    improvements = rag_system(sample_text, mode="improve")

    assert isinstance(improvements, list), "Improvements should be a list"
    assert len(improvements) > 0, "Improvements list is empty"

    print("✔ Improvement suggestions working")
    print(improvements)

    # -----------------------
    # SCORE MODE
    # -----------------------
    print("\n[4] Testing Score Prediction...")

    score = rag_system(sample_text, mode="score")

    assert isinstance(score, float), "Score should be a float"

    print("✔ Score prediction working")
    print(f"Predicted Score: {score}")

    # -----------------------
    # EDGE CASE TEST
    # -----------------------
    print("\n[5] Testing Edge Case (Short Essay)...")

    short_text = "This is a short essay."

    short_feedback = rag_system(short_text, mode="feedback")

    assert len(short_feedback["feedback"]) > 0, "Short essay produced no feedback"

    print("✔ Edge case handled successfully")

    print("\n" + "="*60)
    print("ALL TESTS PASSED ✅")
    print("="*60)

In [24]:
run_sanity_check(df["full_text"].iloc[10])

RUNNING RAG SYSTEM SANITY CHECK

[1] Testing Feedback Mode...
✔ Feedback structure valid
✔ Paragraph generated successfully

----- Generated Paragraph -----
Based on comparable responses, here are some suggestions:

- You have a good foundation—just a few improvements needed.
- Your vocabulary richness (0.52) is below stronger essays (~0.55).
- Using a wider range of vocabulary could improve your essay.


[2] Testing Grade Explanation...
✔ Explanation working
['Your essay is most similar to essays scoring between 3 and 4.', 'The average similarity-based score is approximately 3.40.', 'This suggests your writing level aligns with these examples in both content and structure.']

[3] Testing Improvement Suggestions...
✔ Improvement suggestions working
['No top-tier examples found. Try expanding your essay.']

[4] Testing Score Prediction...
✔ Score prediction working
Predicted Score: 3.4

[5] Testing Edge Case (Short Essay)...
✔ Edge case handled successfully

ALL TESTS PASSED ✅
